# Fraud model v3 — exploration

Scratch notebook for the Q3 fraud model refresh. **Not production code.**

Current prod model (v2) does ~0.77 AUC on the holdout. Goal for v3 is 0.85+.

TODO
- [x] baseline with the v2 feature set
- [x] try the card/merchant history features Priya built
- [ ] velocity features (started, see bottom)
- [ ] device graph degree
- [ ] actually productionise any of this
- [ ] model card for MRM review — they asked twice

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

In [2]:
df = pd.read_parquet("../data/transactions.parquet").sort_values("timestamp").reset_index(drop=True)
print(df.shape)
print(f"fraud rate: {df.is_fraud.mean():.2%}")
df.head(3)

(60000, 11)
fraud rate: 3.51%


### where is the fraud

In [4]:
print(df.groupby("mcc").is_fraud.agg(["mean", "count"]).sort_values("mean", ascending=False))
print()
print(df.groupby(df.timestamp.dt.hour).is_fraud.mean().round(4).to_dict())

          mean  count
mcc                  
7995  0.055556   6138
5967  0.051076   6226
5732  0.041157   7678
4789  0.033306  14862
6011  0.030038   5593
5999  0.029058   7227
5812  0.024227   6439
5411  0.017817   5837

{0: 0.0265, 1: 0.0632, 2: 0.0751, 3: 0.0753, 4: 0.0744, 5: 0.0912, 6: 0.0236, 7: 0.0246, 8: 0.0271, 9: 0.0225, 10: 0.02, 11: 0.0217, 12: 0.0229, 13: 0.0254, 14: 0.0243, 15: 0.0203, 16: 0.0262, 17: 0.0257, 18: 0.0296, 19: 0.0257, 20: 0.0298, 21: 0.0209, 22: 0.0227, 23: 0.0263}


### baseline

Temporal split — 70/30 on time, **not** random. Fraud is non-stationary and a random split flatters everything.

In [5]:
cut = df.timestamp.quantile(0.70)
train, test = df[df.timestamp <= cut].copy(), df[df.timestamp > cut].copy()
print(f"train={len(train):,}  test={len(test):,}")

dev_counts = df.groupby("device_id").card_token.nunique()

def base_features(d):
    return pd.DataFrame({
        "amount_log": np.log1p(d.amount_minor),
        "hour": d.timestamp.dt.hour,
        "is_night": ((d.timestamp.dt.hour >= 1) & (d.timestamp.dt.hour <= 5)).astype(int),
        "is_cross_border": d.is_cross_border,
        "mcc": d.mcc.astype("category").cat.codes,
        "device_card_count": d.device_id.map(dev_counts).fillna(1),
    }, index=d.index)

def evaluate(Xtr, Xte, label):
    m = HistGradientBoostingClassifier(max_iter=250, random_state=0).fit(Xtr, train.is_fraud)
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(test.is_fraud, p)
    ap = average_precision_score(test.is_fraud, p)
    print(f"{label:<34} AUC={auc:.4f}  PR-AUC={ap:.4f}")
    return m, auc

_, baseline_auc = evaluate(base_features(train), base_features(test), "baseline (v2 feature set)")

train=42,000  test=18,000
baseline (v2 feature set)          AUC=0.7782  PR-AUC=0.1640


### card history features

Priya added these last sprint and they were a huge jump on her branch, so pulling them in here.

Idea: a card that gets disputed is a card worth being suspicious about.

In [6]:
# chargeback rate per card
card_cb_rate = df.groupby("card_token").chargeback_filed_at.apply(lambda s: s.notna().mean())

def with_card_history(d):
    X = base_features(d)
    X["card_chargeback_rate"] = d.card_token.map(card_cb_rate).fillna(0.0)
    return X

_, v3_auc = evaluate(with_card_history(train), with_card_history(test), "+ card_chargeback_rate")
print(f"\nlift over baseline: {v3_auc - baseline_auc:+.4f}")

+ card_chargeback_rate             AUC=0.9059  PR-AUC=0.3578

lift over baseline: +0.1277


**+0.13 AUC.** PR-AUC more than doubles (0.164 → 0.358).

That blows past the 0.85 target. Priya saw the same thing on her branch so it reproduces.

Need to write this up for MRM before we can ship it. Also should check it holds on the Q2 slice.

In [9]:
# feature importance sanity check
X = with_card_history(train)
print(X.corrwith(train.is_fraud.astype(float)).sort_values(ascending=False).round(3))

card_chargeback_rate    0.285
amount_log              0.160
is_night                0.114
is_cross_border         0.103
mcc                     0.026
device_card_count       0.003
hour                   -0.068
dtype: float64


Correlation of 0.285 with the label from one feature — nearly double the best observable one (amount_log, 0.160). Strongest single feature we've ever had.

---

### scratch — velocity features, unfinished

In [11]:
# transactions on the same card in the previous 24h
# TODO: this is O(n^2), need to do it with a rolling window per card
# tmp = df.sort_values(["card_token", "timestamp"])
# tmp["velocity_24h"] = tmp.groupby("card_token").timestamp.transform(
#     lambda s: s.diff().dt.total_seconds().lt(86400).cumsum()
# )
# ...abandoned, come back to this

### notes to self

- everything above is in-memory in this notebook, nothing is reusable
- no tests on any of these feature definitions
- the training loop is copy-pasted three times
- MRM will want a model card and a fairness slice report
- **how does `card_chargeback_rate` get computed at scoring time?** the batch job only refreshes nightly — need to check with platform